# [검증] 진재언 위험지수 5변수 재현과 Cronbach's α — 2026-10-01

- 작성자: 이정연 (`ljy`)
- 목적: 충돌 #2(지수 산출 방법론) 검증 — 보고서 서술만으로 5변수를 재현해 Cronbach's α를 독립 계산
- 입력: `data/raw/sk_통신인구_202507~202512_통합.csv` (6개월, outer join 생성본)
- 전제: 집계구=BLOCK_CD 앞 13자리, 대상 시군구=강남구(11230)·춘천시(32010)
- **원 코드 미확인 — 각 변수 정의는 `src/features/risk_axes_v1.py` docstring의 가정을 따름**
- 다루지 않는 범위: 최종 위험지수 가중합, 지역 내부 백분위 정규화, 5분위 구간화

In [1]:
from pathlib import Path

import pandas as pd

from src.features.block_panel import aggregate_to_block
from src.features.risk_axes_v1 import (
    align_risk_direction,
    cronbach_alpha_standardized,
    reduce_to_block_level,
)

MONTHS = ["202507", "202508", "202509", "202510", "202511", "202512"]

repo_root = Path.cwd()
if not (repo_root / "data" / "raw").exists():
    repo_root = repo_root.parent
raw_dir = repo_root / "data" / "raw"

frames = [
    pd.read_csv(
        raw_dir / f"sk_통신인구_{month}_통합.csv",
        dtype={"STD_YM": "string", "BLOCK_CD": "string"},
    )
    for month in MONTHS
]
combined = pd.concat(frames, ignore_index=True)
print(f"격자×월 전체 행: {len(combined):,}")

panel = aggregate_to_block(combined)
print(f"집계구×월 패널 행: {len(panel):,}, 고유 집계구 수: {panel['집계구'].nunique():,}")

격자×월 전체 행: 721,318


집계구×월 패널 행: 9,834, 고유 집계구 수: 1,639


In [2]:
block_level = reduce_to_block_level(panel, months_order=MONTHS)
aligned = align_risk_direction(block_level)

print("집계구 단위 변수 요약:")
print(block_level.describe().round(3))

print("\n방향 정렬 후 상관행렬 (Pearson):")
print(aligned.corr().round(3))

alpha_pearson = cronbach_alpha_standardized(aligned, method="pearson")
alpha_spearman = cronbach_alpha_standardized(aligned, method="spearman")
print(f"\nCronbach's α (Pearson, 표준화): {alpha_pearson:.3f}")
print(f"Cronbach's α (Spearman, 표준화): {alpha_spearman:.3f}")
print(f"항목 수: {aligned.shape[1]}, 유효 집계구 수: {aligned.dropna().shape[0]:,}")

print("\n항목 하나를 뺐을 때의 α (어느 항목이 발목을 잡는지 확인):")
for col in aligned.columns:
    subset = aligned.drop(columns=[col])
    a_p = cronbach_alpha_standardized(subset, method="pearson")
    a_s = cronbach_alpha_standardized(subset, method="spearman")
    print(f"  {col} 제외 → Pearson α={a_p:.3f}, Spearman α={a_s:.3f}")

집계구 단위 변수 요약:
            평탄도      야간비중       주말비      고령비중        추세
count  1639.000  1639.000  1639.000  1639.000  1639.000
mean      0.589     0.055     2.324     0.228    -0.011
std       0.053     0.013     4.031     0.054     0.033
min       0.427     0.020     0.074     0.108    -0.205
25%       0.553     0.046     0.761     0.190    -0.020
50%       0.582     0.054     1.167     0.229    -0.006
75%       0.624     0.063     2.197     0.258     0.005
max       0.901     0.114    68.116     0.575     0.211

방향 정렬 후 상관행렬 (Pearson):
         평탄도_위험  야간비중_위험  주말비_위험  고령비중_위험  추세_위험
평탄도_위험    1.000    0.838   0.286    0.205 -0.020
야간비중_위험   0.838    1.000   0.373    0.146 -0.051
주말비_위험    0.286    0.373   1.000    0.121  0.076
고령비중_위험   0.205    0.146   0.121    1.000  0.252
추세_위험    -0.020   -0.051   0.076    0.252  1.000

Cronbach's α (Pearson, 표준화): 0.589
Cronbach's α (Spearman, 표준화): 0.670
항목 수: 5, 유효 집계구 수: 1,639

항목 하나를 뺐을 때의 α (어느 항목이 발목을 잡는지 확인):
  평탄도_위험 제외 → Pearson α=0.419

## 결론

- **Cronbach's α = 0.589 (Pearson) / 0.670 (Spearman)** — 둘 다 0.7 미만, 기준 미달
- 집계구 단위(1,639개)는 재언 님 보고서의 "1,639개 × 6개월 = 9,834행"과 정확히 일치해, `집계구=BLOCK_CD 앞 13자리`라는 가정 자체는 신뢰할 만하다.
- 상관행렬을 보면 **평탄도_위험 ↔ 야간비중_위험이 0.838**로 사실상 같은 걸 재는 중복 변수다 — 재언 님 보고서 8장이 밝힌 "스피어만 상관 0.89로 이중가중" 문제와 방향·크기가 비슷하다(다만 우리가 지금 비교한 건 재언 님의 Spearman 0.89가 아니라 우리 Pearson 0.838이다 — 같은 조합으로 Spearman을 다시 재면 0.843으로, 역시 비슷한 크기다). 보고서는 이 중복을 가중치에서만 뺐지, **5개 전체의 α를 재검정한 기록은 없었다** — 지금 돌린 게 그 재검정이다.
- **추세_위험은 평탄도·야간비중과는 거의 무관하지만(-0.020, -0.051), 고령비중과는 약한 양의 상관(0.252)이 있다** — 나머지와 "완전히 무관"하지는 않다.
- **주말비_위험**은 평탄도·야간비중과 약한 양의 상관(0.29~0.37), 고령비중·추세와는 거의 무관(0.12, 0.08) — 충돌 #3에서 의심했던 "방향이 맞는지 불확실하다"는 우려와 일관된 패턴이다.
- **항목 하나를 뺐을 때 α가 어떻게 바뀌는지**(Pearson 기준):

  | 제외한 항목 | α |
  | --- | ---: |
  | 평탄도_위험 | 0.419 (↓) |
  | 야간비중_위험 | 0.420 (↓) |
  | 주말비_위험 | 0.542 (↓) |
  | 고령비중_위험 | 0.572 (↓) |
  | **추세_위험** | **0.661 (↑, 최대)** |
  | (아무것도 안 뺌) | 0.589 |

  **평탄도·야간비중 중 하나를 빼면 α가 오히려 0.42 안팎으로 더 떨어진다** — 이 둘을 중복으로 보고 하나를 빼는 건 이중가중 문제를 고치는 것이지 내적 일관성(α)을 올리는 해법이 아니다. α를 가장 많이 올리는 조치는 **추세를 빼는 것**이고(0.589→0.661), 그래도 0.7에는 못 미친다.
- **결론**: 충돌 #2는 "역시 문제가 있었다" 쪽으로 기운다. 김태은 1차 설계(α −4.219)만큼 심각하진 않지만, 팀이 쓰는 0.7 기준에는 Pearson·Spearman 둘 다 못 미친다. 추세를 빼거나 재정의하는 게 α를 가장 많이 올리지만 그것만으로는 기준을 넘지 못하므로, 다섯 변수 자체를 재검토(특히 평탄도/야간비중 중복, 주말비 방향)할 필요가 있다.
- **다음 확인 필요**: 이 계산 자체가 원 코드 미확인 상태의 재구성이므로, 재언 님 실제 코드가 공유되면 (a) 집계구 정의, (b) 주말비 방향, (c) 평탄도/야간비중/추세의 정확한 공식을 대조해 이 α 값을 재검증해야 한다.